In [10]:
"""
Script xử lý raster ĐBSCL - Chạy trên VSCode
Cài đặt: pip install gdal rasterio geopandas matplotlib pillow numpy
"""

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import rasterio
from rasterio.mask import mask
from rasterio.plot import show
import geopandas as gpd
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ====================================
# CẤU HÌNH - THAY ĐỔI CHO PHÙ HỢP
# ====================================


INPUT_FOLDER = r"E:\DownloadData\co2_ban_do\co2_map_new\map_stacking\RRD"
SHAPEFILE_PATH = r"E:\RanhGioi\DBSH_utm\DBSH.shp"
OUTPUT_FOLDER = r"E:\DownloadData\co2_ban_do\output_new\RRD\Mua_1"
PATTERN = "CO2_RRD_*_Mua.tif"

# Tạo các thư mục con
CLIPPED_FOLDER = os.path.join(OUTPUT_FOLDER, "1_clipped")
STYLED_FOLDER = os.path.join(OUTPUT_FOLDER, "2_styled_images")
TIMESERIES_FOLDER = os.path.join(OUTPUT_FOLDER, "3_timeseries")

# ====================================
# BƯỚC 1: CẮT FILE TIF THEO RANH GIỚI
# ====================================
def clip_rasters_by_boundary(input_folder, pattern, shapefile_path, output_folder):
    """
    Cắt tất cả raster theo ranh giới shapefile
    """
    print("\n" + "="*60)
    print("BƯỚC 1: CẮT RASTER THEO RANH GIỚI ĐBSCL")
    print("="*60)
    
    os.makedirs(output_folder, exist_ok=True)
    
    # Đọc shapefile
    gdf = gpd.read_file(shapefile_path)
    
    # Chuyển sang dict để dùng với rasterio
    shapes = [feature["geometry"] for feature in gdf.__geo_interface__['features']]
    
    # Tìm tất cả file raster
    raster_files = sorted(glob.glob(os.path.join(input_folder, pattern)))
    print(f"Tìm thấy {len(raster_files)} file raster")
    
    clipped_files = []
    
    for idx, raster_path in enumerate(raster_files, 1):
        filename = os.path.basename(raster_path)
        output_path = os.path.join(output_folder, filename.replace('.tif', '_clipped.tif'))
        
        print(f"\n[{idx}/{len(raster_files)}] Đang cắt: {filename}")
        
        try:
            with rasterio.open(raster_path) as src:
                # Cắt raster theo shapefile
                out_image, out_transform = mask(src, shapes, crop=True, nodata=-9999)
                out_meta = src.meta.copy()
                
                # Cập nhật metadata
                out_meta.update({
                    "driver": "GTiff",
                    "height": out_image.shape[1],
                    "width": out_image.shape[2],
                    "transform": out_transform,
                    "nodata": -9999,
                    "compress": "lzw"
                })
                
                # Lưu file
                with rasterio.open(output_path, "w", **out_meta) as dest:
                    dest.write(out_image)
                
                clipped_files.append(output_path)
                print(f"   ✓ Đã lưu: {output_path}")
                
        except Exception as e:
            print(f"   ✗ Lỗi: {str(e)}")
            continue
    
    print(f"\n✓ Hoàn thành cắt {len(clipped_files)} file")
    return clipped_files, gdf

# ====================================
# BƯỚC 2: TẠO COLOR MAP ĐỎ
# ====================================
def create_red_colormap():
    """
    Tạo colormap từ đỏ nhạt đến đỏ đậm
    """
    colors = [
        '#ffcccc',  # Đỏ rất nhạt
        '#ffaaaa',
        '#ff8888',
        '#ff6666',
        '#ff4444',
        '#ff2222',
        '#ee0000',
        '#cc0000',
        '#aa0000',
        '#8b0000'   # Đỏ rất đậm
    ]
    n_bins = 256
    cmap = LinearSegmentedColormap.from_list('red_gradient', colors, N=n_bins)
    return cmap

# ====================================
# BƯỚC 3: VẼ VÀ LƯU TỪNG ẢNH CÓ RANH GIỚI
# ====================================
def plot_and_save_single_raster(raster_path, boundary_gdf, output_path, cmap, dpi=300):
    """
    Vẽ một raster với ranh giới và lưu ảnh
    """
    with rasterio.open(raster_path) as src:
        data = src.read(1)
        
        # Tính min/max, bỏ nodata
        nodata = src.nodata
        valid_data = data[data != nodata] if nodata else data.flatten()
        
        if len(valid_data) == 0:
            print(f"   ⚠ Không có dữ liệu hợp lệ trong {os.path.basename(raster_path)}")
            return
        
        vmin, vmax = np.percentile(valid_data, [2, 98])
        
        # Tạo figure
        fig, ax = plt.subplots(figsize=(14, 12))
        
        # Vẽ raster
        show(src, ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, interpolation='bilinear')
        
        # Vẽ ranh giới shapefile
        boundary_gdf.boundary.plot(ax=ax, color='black', linewidth=2, zorder=2)
        
        # Tiêu đề
        title = os.path.basename(raster_path).replace('_clipped.tif', '').replace('_', ' ')
        ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
        
        # Thêm colorbar
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, orientation='vertical')
        cbar.set_label('CO₂ Value', rotation=270, labelpad=25, fontsize=14)
        cbar.ax.tick_params(labelsize=11)
        
        # Tùy chỉnh trục
        ax.set_xlabel('Longitude', fontsize=13, fontweight='bold')
        ax.set_ylabel('Latitude', fontsize=13, fontweight='bold')
        ax.tick_params(labelsize=11)
        ax.set_aspect('equal')
        
        # Grid nhẹ
        ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
        
        plt.tight_layout()
        plt.savefig(output_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        plt.close()

def export_styled_images(clipped_files, boundary_gdf, output_folder):
    """
    Export tất cả raster thành ảnh PNG với ranh giới
    """
    print("\n" + "="*60)
    print("BƯỚC 2+3: ĐỔI MÀU VÀ XUẤT ẢNH CÓ RANH GIỚI")
    print("="*60)
    
    os.makedirs(output_folder, exist_ok=True)
    
    cmap = create_red_colormap()
    
    for idx, raster_path in enumerate(clipped_files, 1):
        filename = os.path.basename(raster_path).replace('_clipped.tif', '.png')
        output_path = os.path.join(output_folder, filename)
        
        print(f"\n[{idx}/{len(clipped_files)}] Đang vẽ: {filename}")
        
        try:
            plot_and_save_single_raster(raster_path, boundary_gdf, output_path, cmap)
            print(f"   ✓ Đã lưu: {output_path}")
        except Exception as e:
            print(f"   ✗ Lỗi: {str(e)}")
            continue
    
    print(f"\n✓ Hoàn thành export {len(clipped_files)} ảnh")

# ====================================
# BƯỚC 4: TẠO TIME-SERIES COMPARISON
# ====================================
def create_timeseries_comparison(clipped_files, boundary_gdf, output_folder):
    """
    Tạo bảng so sánh time-series nhiều năm
    """
    print("\n" + "="*60)
    print("BƯỚC 4: TẠO TIME-SERIES COMPARISON")
    print("="*60)
    
    os.makedirs(output_folder, exist_ok=True)
    
    n_maps = len(clipped_files)
    
    # Tính layout
    if n_maps <= 4:
        rows, cols = 2, 2
    elif n_maps <= 6:
        rows, cols = 2, 3
    elif n_maps <= 9:
        rows, cols = 3, 3
    else:
        rows, cols = 4, 3
    
    # Tạo colormap
    cmap = create_red_colormap()
    
    # Tìm global min/max để đồng nhất màu sắc
    print("\nĐang tính global min/max...")
    global_min, global_max = np.inf, -np.inf
    
    for raster_path in clipped_files:
        with rasterio.open(raster_path) as src:
            data = src.read(1)
            nodata = src.nodata
            valid_data = data[data != nodata] if nodata else data.flatten()
            
            if len(valid_data) > 0:
                vmin, vmax = np.percentile(valid_data, [2, 98])
                global_min = min(global_min, vmin)
                global_max = max(global_max, vmax)
    
    print(f"Global range: {global_min:.2f} - {global_max:.2f}")
    
    # Tạo figure lớn
    fig = plt.figure(figsize=(cols * 7, rows * 6))
    
    # Vẽ từng subplot
    for idx, raster_path in enumerate(clipped_files[:rows*cols]):
        ax = plt.subplot(rows, cols, idx + 1)
        
        print(f"Đang vẽ subplot {idx+1}/{min(n_maps, rows*cols)}...")
        
        try:
            with rasterio.open(raster_path) as src:
                show(src, ax=ax, cmap=cmap, vmin=global_min, vmax=global_max, 
                     interpolation='bilinear')
                
                # Vẽ ranh giới
                boundary_gdf.boundary.plot(ax=ax, color='black', linewidth=1.5, zorder=2)
                
                # Tiêu đề
                year = os.path.basename(raster_path).replace('CO2_MKD_', '').replace('_DongXuan_clipped.tif', '')
                ax.set_title(f'Năm {year}', fontsize=14, fontweight='bold', pad=10)
                ax.set_aspect('equal')
                ax.tick_params(labelsize=9)
                ax.grid(True, alpha=0.2, linestyle='--', linewidth=0.3)
                
        except Exception as e:
            print(f"   ✗ Lỗi subplot {idx+1}: {str(e)}")
            ax.text(0.5, 0.5, 'Error', ha='center', va='center', transform=ax.transAxes)
    
    # Thêm colorbar chung
    fig.subplots_adjust(right=0.92, hspace=0.3, wspace=0.3)
    cbar_ax = fig.add_axes([0.94, 0.15, 0.02, 0.7])
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=global_min, vmax=global_max))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label('CO₂ Value', rotation=270, labelpad=30, fontsize=16, fontweight='bold')
    cbar.ax.tick_params(labelsize=12)
    
    # Title tổng
    fig.suptitle('SO SÁNH CO₂ ĐỒNG BẰNG SÔNG CỬU LONG THEO NĂM - VỤ ĐÔNG XUÂN', 
                 fontsize=20, fontweight='bold', y=0.98)
    
    # Lưu file
    output_path = os.path.join(output_folder, 'DBSCL_TimeSeries_Comparison.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"\n✓ Đã lưu time-series comparison: {output_path}")

# ====================================
# BONUS: TẠO GIF ANIMATION
# ====================================
def create_gif_animation(styled_folder, output_folder, duration=1000):
    """
    Tạo GIF animation từ các ảnh PNG
    """
    print("\n" + "="*60)
    print("BONUS: TẠO GIF ANIMATION")
    print("="*60)
    
    png_files = sorted(glob.glob(os.path.join(styled_folder, "*.png")))
    
    if len(png_files) < 2:
        print("Cần ít nhất 2 ảnh để tạo GIF")
        return
    
    images = []
    for png_file in png_files:
        images.append(Image.open(png_file))
    
    output_gif = os.path.join(output_folder, 'DBSCL_TimeSeries_Animation.gif')
    
    images[0].save(
        output_gif,
        save_all=True,
        append_images=images[1:],
        duration=duration,
        loop=0,
        optimize=True
    )
    
    print(f"✓ Đã tạo GIF animation: {output_gif}")
    print(f"  Số frame: {len(images)}, Duration: {duration}ms/frame")

# ====================================
# HÀM CHÍNH
# ====================================
def main():
    print("\n" + "="*70)
    print("SCRIPT XỬ LÝ RASTER ĐBSCL - CHẠY TRÊN VSCODE")
    print("="*70)
    print(f"Input folder: {INPUT_FOLDER}")
    print(f"Shapefile: {SHAPEFILE_PATH}")
    print(f"Output folder: {OUTPUT_FOLDER}")
    print("="*70)
    
    try:
        # Bước 1: Cắt raster theo ranh giới
        clipped_files, boundary_gdf = clip_rasters_by_boundary(
            INPUT_FOLDER, 
            PATTERN, 
            SHAPEFILE_PATH, 
            CLIPPED_FOLDER
        )
        
        if not clipped_files:
            print("\n✗ Không có file nào được cắt thành công!")
            return
        
        # Bước 2+3: Đổi màu và export ảnh
        export_styled_images(clipped_files, boundary_gdf, STYLED_FOLDER)
        
        # Bước 4: Tạo time-series comparison
        create_timeseries_comparison(clipped_files, boundary_gdf, TIMESERIES_FOLDER)
        
        # Bonus: Tạo GIF animation
        create_gif_animation(STYLED_FOLDER, TIMESERIES_FOLDER, duration=1000)
        
        print("\n" + "="*70)
        print("✓✓✓ HOÀN THÀNH TẤT CẢ CÁC BƯỚC ✓✓✓")
        print("="*70)
        print(f"\nKết quả được lưu tại:")
        print(f"  1. Raster đã cắt:    {CLIPPED_FOLDER}")
        print(f"  2. Ảnh đã style:     {STYLED_FOLDER}")
        print(f"  3. Time-series:      {TIMESERIES_FOLDER}")
        print(f"     - PNG comparison")
        print(f"     - GIF animation")
        
    except Exception as e:
        print(f"\n✗✗✗ LỖI: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()


SCRIPT XỬ LÝ RASTER ĐBSCL - CHẠY TRÊN VSCODE
Input folder: E:\DownloadData\co2_ban_do\co2_map_new\map_stacking\RRD
Shapefile: E:\RanhGioi\DBSH_utm\DBSH.shp
Output folder: E:\DownloadData\co2_ban_do\output_new\RRD\Mua_1

BƯỚC 1: CẮT RASTER THEO RANH GIỚI ĐBSCL
Tìm thấy 5 file raster

[1/5] Đang cắt: CO2_RRD_2020_Mua.tif
   ✓ Đã lưu: E:\DownloadData\co2_ban_do\output_new\RRD\Mua_1\1_clipped\CO2_RRD_2020_Mua_clipped.tif

[2/5] Đang cắt: CO2_RRD_2021_Mua.tif
   ✓ Đã lưu: E:\DownloadData\co2_ban_do\output_new\RRD\Mua_1\1_clipped\CO2_RRD_2021_Mua_clipped.tif

[3/5] Đang cắt: CO2_RRD_2022_Mua.tif
   ✓ Đã lưu: E:\DownloadData\co2_ban_do\output_new\RRD\Mua_1\1_clipped\CO2_RRD_2022_Mua_clipped.tif

[4/5] Đang cắt: CO2_RRD_2023_Mua.tif
   ✓ Đã lưu: E:\DownloadData\co2_ban_do\output_new\RRD\Mua_1\1_clipped\CO2_RRD_2023_Mua_clipped.tif

[5/5] Đang cắt: CO2_RRD_2024_Mua.tif
   ✓ Đã lưu: E:\DownloadData\co2_ban_do\output_new\RRD\Mua_1\1_clipped\CO2_RRD_2024_Mua_clipped.tif

✓ Hoàn thành cắt 5 file



In [1]:
"""
Script xử lý raster ĐBSCL - Chạy trên VSCode
Cài đặt: pip install gdal rasterio geopandas matplotlib pillow numpy
"""

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import rasterio
from rasterio.mask import mask
from rasterio.plot import show
import geopandas as gpd
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ====================================
# CẤU HÌNH - THAY ĐỔI CHO PHÙ HỢP
# ====================================

INPUT_FOLDER = r"E:\DownloadData\co2_ban_do\co2_map_new\map_stacking\RRD"
SHAPEFILE_PATH = r"E:\RanhGioi\DBSH_utm\DBSH.shp"
OUTPUT_FOLDER = r"E:\DownloadData\co2_ban_do\output_new\RRD\Mua_4"
PATTERN = "CO2_RRD_*_Mua.tif"

# THANG GIÁ TRỊ CỐ ĐỊNH (sửa theo dữ liệu của bạn)
USE_FIXED_SCALE = True  # True = dùng thang cố định, False = tự động theo từng ảnh
FIXED_VMIN = 419.4     # Giá trị min hiển thị trên colorbar
FIXED_VMAX = 420.1      # Giá trị max hiển thị trên colorbar
EXTEND_COLORBAR = 'both'  # 'both' = mở rộng 2 đầu, 'min' = chỉ dưới, 'max' = chỉ trên, 'neither' = không mở rộng

# Tạo các thư mục con
CLIPPED_FOLDER = os.path.join(OUTPUT_FOLDER, "1_clipped")
STYLED_FOLDER = os.path.join(OUTPUT_FOLDER, "2_styled_images")
TIMESERIES_FOLDER = os.path.join(OUTPUT_FOLDER, "3_timeseries")

# ====================================
# BƯỚC 1: CẮT FILE TIF THEO RANH GIỚI
# ====================================
def clip_rasters_by_boundary(input_folder, pattern, shapefile_path, output_folder):
    """
    Cắt tất cả raster theo ranh giới shapefile
    """
    print("\n" + "="*60)
    print("BƯỚC 1: CẮT RASTER THEO RANH GIỚI ĐBSCL")
    print("="*60)
    
    os.makedirs(output_folder, exist_ok=True)
    
    # Đọc shapefile
    gdf = gpd.read_file(shapefile_path)
    
    # Chuyển sang dict để dùng với rasterio
    shapes = [feature["geometry"] for feature in gdf.__geo_interface__['features']]
    
    # Tìm tất cả file raster
    raster_files = sorted(glob.glob(os.path.join(input_folder, pattern)))
    print(f"Tìm thấy {len(raster_files)} file raster")
    
    clipped_files = []
    
    for idx, raster_path in enumerate(raster_files, 1):
        filename = os.path.basename(raster_path)
        output_path = os.path.join(output_folder, filename.replace('.tif', '_clipped.tif'))
        
        print(f"\n[{idx}/{len(raster_files)}] Đang cắt: {filename}")
        
        try:
            with rasterio.open(raster_path) as src:
                # Cắt raster theo shapefile
                out_image, out_transform = mask(src, shapes, crop=True, nodata=-9999)
                out_meta = src.meta.copy()
                
                # Cập nhật metadata
                out_meta.update({
                    "driver": "GTiff",
                    "height": out_image.shape[1],
                    "width": out_image.shape[2],
                    "transform": out_transform,
                    "nodata": -9999,
                    "compress": "lzw"
                })
                
                # Lưu file
                with rasterio.open(output_path, "w", **out_meta) as dest:
                    dest.write(out_image)
                
                clipped_files.append(output_path)
                print(f"   ✓ Đã lưu: {output_path}")
                
        except Exception as e:
            print(f"   ✗ Lỗi: {str(e)}")
            continue
    
    print(f"\n✓ Hoàn thành cắt {len(clipped_files)} file")
    return clipped_files, gdf

# ====================================
# BƯỚC 2: TẠO COLOR MAP ĐỎ
# ====================================
def create_red_colormap():
    """
    Tạo colormap từ đỏ nhạt đến đỏ đậm
    """
    colors = [
        '#ffcccc',  # Đỏ rất nhạt
        '#ffaaaa',
        '#ff8888',
        '#ff6666',
        '#ff4444',
        '#ff2222',
        '#ee0000',
        '#cc0000',
        '#aa0000',
        '#8b0000'   # Đỏ rất đậm
    ]
    n_bins = 256
    cmap = LinearSegmentedColormap.from_list('red_gradient', colors, N=n_bins)
    return cmap

# ====================================
# BƯỚC 3: VẼ VÀ LƯU TỪNG ẢNH CÓ RANH GIỚI
# ====================================
def plot_and_save_single_raster(raster_path, boundary_gdf, output_path, cmap, vmin=None, vmax=None, dpi=300):
    """
    Vẽ một raster với ranh giới và lưu ảnh
    vmin, vmax: nếu None thì tự động tính, nếu có giá trị thì dùng giá trị cố định
    """
    with rasterio.open(raster_path) as src:
        data = src.read(1)
        
        # Nếu không có vmin/vmax được chỉ định, tính tự động
        if vmin is None or vmax is None:
            nodata = src.nodata
            valid_data = data[data != nodata] if nodata else data.flatten()
            
            if len(valid_data) == 0:
                print(f"   ⚠ Không có dữ liệu hợp lệ trong {os.path.basename(raster_path)}")
                return
            
            vmin, vmax = np.percentile(valid_data, [2, 98])
        
        # Tạo figure
        fig, ax = plt.subplots(figsize=(14, 12))
        
        # Vẽ raster
        show(src, ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, interpolation='bilinear')
        
        # Vẽ ranh giới shapefile
        boundary_gdf.boundary.plot(ax=ax, color='black', linewidth=2, zorder=2)
        
        # Tiêu đề
        title = os.path.basename(raster_path).replace('_clipped.tif', '').replace('_', ' ')
        ax.set_title(title, fontsize=20, fontweight='bold', pad=20)
        
        # Thêm colorbar với extend
        extend = EXTEND_COLORBAR if USE_FIXED_SCALE else 'neither'
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, orientation='vertical', extend=extend)
        cbar.set_label('CO₂ Value', rotation=270, labelpad=25, fontsize=14)
        cbar.ax.tick_params(labelsize=11)
        
        # Tùy chỉnh trục
        ax.set_xlabel('Longitude', fontsize=14, fontweight='bold')
        ax.set_ylabel('Latitude', fontsize=14, fontweight='bold')
        ax.tick_params(labelsize=11)
        ax.set_aspect('equal')
        
        # Grid nhẹ
        ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
        
        plt.tight_layout()
        plt.savefig(output_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        plt.close()

def export_styled_images(clipped_files, boundary_gdf, output_folder):
    """
    Export tất cả raster thành ảnh PNG với ranh giới
    """
    print("\n" + "="*60)
    print("BƯỚC 2+3: ĐỔI MÀU VÀ XUẤT ẢNH CÓ RANH GIỚI")
    print("="*60)
    
    os.makedirs(output_folder, exist_ok=True)
    
    cmap = create_red_colormap()
    
    # Xác định vmin/vmax
    if USE_FIXED_SCALE:
        vmin, vmax = FIXED_VMIN, FIXED_VMAX
        print(f"\n📊 Sử dụng thang giá trị CỐ ĐỊNH: {vmin} - {vmax}")
    else:
        vmin, vmax = None, None
        print(f"\n📊 Sử dụng thang giá trị TỰ ĐỘNG cho từng ảnh")
    
    for idx, raster_path in enumerate(clipped_files, 1):
        filename = os.path.basename(raster_path).replace('_clipped.tif', '.png')
        output_path = os.path.join(output_folder, filename)
        
        print(f"\n[{idx}/{len(clipped_files)}] Đang vẽ: {filename}")
        
        try:
            plot_and_save_single_raster(raster_path, boundary_gdf, output_path, cmap, vmin, vmax)
            print(f"   ✓ Đã lưu: {output_path}")
        except Exception as e:
            print(f"   ✗ Lỗi: {str(e)}")
            continue
    
    print(f"\n✓ Hoàn thành export {len(clipped_files)} ảnh")

# ====================================
# BƯỚC 4: TẠO TIME-SERIES COMPARISON
# ====================================
def create_timeseries_comparison(clipped_files, boundary_gdf, output_folder):
    """
    Tạo bảng so sánh time-series nhiều năm
    """
    print("\n" + "="*60)
    print("BƯỚC 4: TẠO TIME-SERIES COMPARISON")
    print("="*60)
    
    os.makedirs(output_folder, exist_ok=True)
    
    n_maps = len(clipped_files)
    
    # Tính layout
    if n_maps <= 4:
        rows, cols = 2, 2
    elif n_maps <= 6:
        rows, cols = 2, 3
    elif n_maps <= 9:
        rows, cols = 3, 3
    else:
        rows, cols = 4, 3
    
    # Tạo colormap
    cmap = create_red_colormap()
    
    # Xác định min/max
    if USE_FIXED_SCALE:
        global_min, global_max = FIXED_VMIN, FIXED_VMAX
        print(f"\n📊 Sử dụng thang giá trị CỐ ĐỊNH: {global_min} - {global_max}")
    else:
        # Tìm global min/max tự động
        print("\nĐang tính global min/max tự động...")
        global_min, global_max = np.inf, -np.inf
        
        for raster_path in clipped_files:
            with rasterio.open(raster_path) as src:
                data = src.read(1)
                nodata = src.nodata
                valid_data = data[data != nodata] if nodata else data.flatten()
                
                if len(valid_data) > 0:
                    vmin, vmax = np.percentile(valid_data, [2, 98])
                    global_min = min(global_min, vmin)
                    global_max = max(global_max, vmax)
        
        print(f"Global range tự động: {global_min:.2f} - {global_max:.2f}")
    
    # Tạo figure lớn
    fig = plt.figure(figsize=(cols * 7, rows * 6))
    
    # Vẽ từng subplot
    for idx, raster_path in enumerate(clipped_files[:rows*cols]):
        ax = plt.subplot(rows, cols, idx + 1)
        
        print(f"Đang vẽ subplot {idx+1}/{min(n_maps, rows*cols)}...")
        
        try:
            with rasterio.open(raster_path) as src:
                show(src, ax=ax, cmap=cmap, vmin=global_min, vmax=global_max, 
                     interpolation='bilinear')
                
                # Vẽ ranh giới
                boundary_gdf.boundary.plot(ax=ax, color='black', linewidth=1.5, zorder=2)
                
                # Tiêu đề
                year = os.path.basename(raster_path).replace('CO2_MKD_', '').replace('_DongXuan_clipped.tif', '')
                ax.set_title(f'Năm {year}', fontsize=16, fontweight='bold', pad=10)
                ax.set_aspect('equal')
                ax.tick_params(labelsize=9)
                ax.grid(True, alpha=0.2, linestyle='--', linewidth=0.3)
                
        except Exception as e:
            print(f"   ✗ Lỗi subplot {idx+1}: {str(e)}")
            ax.text(0.5, 0.5, 'Error', ha='center', va='center', transform=ax.transAxes)
    
    # Thêm colorbar chung với extend
    fig.subplots_adjust(right=0.92, hspace=0.3, wspace=0.3)
    cbar_ax = fig.add_axes([0.94, 0.15, 0.02, 0.7])
    extend = EXTEND_COLORBAR if USE_FIXED_SCALE else 'neither'
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=global_min, vmax=global_max))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax, extend=extend)
    cbar.set_label('CO₂ Value', rotation=270, labelpad=30, fontsize=16, fontweight='bold')
    cbar.ax.tick_params(labelsize=12)
    
    # Title tổng
    scale_info = f"(Thang cố định: {global_min:.1f} - {global_max:.1f})" if USE_FIXED_SCALE else "(Thang tự động)"
    fig.suptitle(f'SO SÁNH CO₂ ĐỒNG BẰNG SÔNG HỒNG THEO NĂM - VỤ MÙA', 
                 fontsize=20, fontweight='bold', y=0.98)
    
    # Lưu file
    output_path = os.path.join(output_folder, 'DBSH_TimeSeries_Comparison.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"\n✓ Đã lưu time-series comparison: {output_path}")

# ====================================
# BONUS: TẠO GIF ANIMATION
# ====================================
def create_gif_animation(styled_folder, output_folder, duration=1000):
    """
    Tạo GIF animation từ các ảnh PNG
    """
    print("\n" + "="*60)
    print("BONUS: TẠO GIF ANIMATION")
    print("="*60)
    
    png_files = sorted(glob.glob(os.path.join(styled_folder, "*.png")))
    
    if len(png_files) < 2:
        print("Cần ít nhất 2 ảnh để tạo GIF")
        return
    
    images = []
    for png_file in png_files:
        images.append(Image.open(png_file))
    
    output_gif = os.path.join(output_folder, 'DBSH_TimeSeries_Animation.gif')
    
    images[0].save(
        output_gif,
        save_all=True,
        append_images=images[1:],
        duration=duration,
        loop=0,
        optimize=True
    )
    
    print(f"✓ Đã tạo GIF animation: {output_gif}")
    print(f"  Số frame: {len(images)}, Duration: {duration}ms/frame")

# ====================================
# HÀM CHÍNH
# ====================================
def main():
    print("\n" + "="*70)
    print("SCRIPT XỬ LÝ RASTER ĐBSCL - CHẠY TRÊN VSCODE")
    print("="*70)
    print(f"Input folder: {INPUT_FOLDER}")
    print(f"Shapefile: {SHAPEFILE_PATH}")
    print(f"Output folder: {OUTPUT_FOLDER}")
    print("="*70)
    
    try:
        # Bước 1: Cắt raster theo ranh giới
        clipped_files, boundary_gdf = clip_rasters_by_boundary(
            INPUT_FOLDER, 
            PATTERN, 
            SHAPEFILE_PATH, 
            CLIPPED_FOLDER
        )
        
        if not clipped_files:
            print("\n✗ Không có file nào được cắt thành công!")
            return
        
        # Bước 2+3: Đổi màu và export ảnh
        export_styled_images(clipped_files, boundary_gdf, STYLED_FOLDER)
        
        # Bước 4: Tạo time-series comparison
        create_timeseries_comparison(clipped_files, boundary_gdf, TIMESERIES_FOLDER)
        
        # Bonus: Tạo GIF animation
        create_gif_animation(STYLED_FOLDER, TIMESERIES_FOLDER, duration=1000)
        
        print("\n" + "="*70)
        print("✓✓✓ HOÀN THÀNH TẤT CẢ CÁC BƯỚC ✓✓✓")
        print("="*70)
        print(f"\nKết quả được lưu tại:")
        print(f"  1. Raster đã cắt:    {CLIPPED_FOLDER}")
        print(f"  2. Ảnh đã style:     {STYLED_FOLDER}")
        print(f"  3. Time-series:      {TIMESERIES_FOLDER}")
        print(f"     - PNG comparison")
        print(f"     - GIF animation")
        
    except Exception as e:
        print(f"\n✗✗✗ LỖI: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()


SCRIPT XỬ LÝ RASTER ĐBSCL - CHẠY TRÊN VSCODE
Input folder: E:\DownloadData\co2_ban_do\co2_map_new\map_stacking\RRD
Shapefile: E:\RanhGioi\DBSH_utm\DBSH.shp
Output folder: E:\DownloadData\co2_ban_do\output_new\RRD\Mua_4

BƯỚC 1: CẮT RASTER THEO RANH GIỚI ĐBSCL
Tìm thấy 5 file raster

[1/5] Đang cắt: CO2_RRD_2020_Mua.tif
   ✓ Đã lưu: E:\DownloadData\co2_ban_do\output_new\RRD\Mua_4\1_clipped\CO2_RRD_2020_Mua_clipped.tif

[2/5] Đang cắt: CO2_RRD_2021_Mua.tif
   ✓ Đã lưu: E:\DownloadData\co2_ban_do\output_new\RRD\Mua_4\1_clipped\CO2_RRD_2021_Mua_clipped.tif

[3/5] Đang cắt: CO2_RRD_2022_Mua.tif
   ✓ Đã lưu: E:\DownloadData\co2_ban_do\output_new\RRD\Mua_4\1_clipped\CO2_RRD_2022_Mua_clipped.tif

[4/5] Đang cắt: CO2_RRD_2023_Mua.tif
   ✓ Đã lưu: E:\DownloadData\co2_ban_do\output_new\RRD\Mua_4\1_clipped\CO2_RRD_2023_Mua_clipped.tif

[5/5] Đang cắt: CO2_RRD_2024_Mua.tif
   ✓ Đã lưu: E:\DownloadData\co2_ban_do\output_new\RRD\Mua_4\1_clipped\CO2_RRD_2024_Mua_clipped.tif

✓ Hoàn thành cắt 5 file

